In [1]:
import sys
sys.path.insert(0, '../')
from data.Reinhard import Reinhard
from models.model_mrcnn import _default_mrcnn_config, build_default
from visualization.explain import ExplainPredictions
from glob import  glob
import os
import pandas as pd
import numpy as np
from PIL import Image

In [2]:
#LBD_model_path = '/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/models/mrcnn_models/fast-bush-229_mrcnn_model_75.pth'
WM_model_path = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/WM_models/3uctrn1o10.pth'
LBD_model_path = '/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/models/mrcnn_models/woven-wood-253_mrcnn_model_24.pth'
#save_results_path="/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/WGM_crops_gt_predicted_LBs/"+LBD_model_path.split("/")[-1]+"_"+WM_model_path.split("/")[-1]
save_results_path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops_pred_LBs/"+LBD_model_path.split("/")[-1]+"_"+WM_model_path.split("/")[-1]+"_updated_code"
if not os.path.exists(save_results_path):
    os.makedirs(save_results_path)
test_config = dict(
        batch_size = 1,
        num_classes = 2
    )

model_config = _default_mrcnn_config(num_classes=1 + test_config['num_classes']).config
model_lbd = build_default(model_config, im_size=1024)

def run_lb_seg(img_crop_list, folder_name):
    final_df = pd.DataFrame()
    results_path = os.path.join(save_results_path,folder_name.split("/")[-1],"results")
    obj_coords_all = []
    if not os.path.exists(results_path):
        os.makedirs(results_path)
    masks_path =  os.path.join(save_results_path, folder_name.split("/")[-1],"masks")
    if not os.path.exists(masks_path):
        os.makedirs(masks_path)
    detections_path = os.path.join( save_results_path, folder_name.split("/")[-1],"detections")
    if not os.path.exists(detections_path):
        os.makedirs(detections_path)
        
    for img in img_crop_list:
        img_name =  img.split("/")[-1].replace(".svs","")
        image_array = np.array(Image.open(img))
        explain= ExplainPredictions(model_lbd, model_input_path = LBD_model_path, test_input_path=[image_array], 
                                        detection_threshold=0.65, wandb='', save_result=False, ablation_cam=False, save_thresholds=False,
                                        results_path=results_path,masks_path=masks_path, detections_path=detections_path, img_name = img_name)
        
        detected_img_list, boxes_list, df, obj_coords1 = explain.generate_results_v1()
        if len(final_df)==0:
            final_df = df
        else:
            final_df = pd.concat([final_df, df])
        x = int(float(img_name.split("_x_")[1].split("_")[0]))
        y = int(float(img_name.split("_y_")[1].split("_")[0].split(".")[0]))
        obj_coords1 = [[x-256 + i[0], y+i[1]] for i in obj_coords1]
        obj_coords_all.extend(obj_coords1)
    final_df.to_csv(os.path.join(save_results_path, folder_name.split("/")[-1], folder_name.split("/")[-1].split(".")[0]+".csv"))
    pd.DataFrame(obj_coords_all, columns=["x","y"]).to_csv(os.path.join(save_results_path, folder_name.split("/")[-1], folder_name.split("/")[-1].split(".")[0]+"_coords"+".csv"),index=False)

    return obj_coords_all

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [40]:
obj_coords = []
folder_name = wsi
for img in img_crop_list:
    img_name =  img.split("/")[-1].replace(".svs","")
    x = int(float(img_name.split("_x_")[1].split("_")[0]))
    y = int(float(img_name.split("_y_")[1].split("_")[0].split(".")[0]))
    obj_coords.append([x,y])
pd.DataFrame(obj_coords, columns=["x","y"]).to_csv(os.path.join(save_results_path, folder_name.split("/")[-1], folder_name.split("/")[-1].split(".")[0]+"_coords"+".csv"),index=False)

    

In [3]:
#wsi_list = glob(os.path.join("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops1/3uctrn1o10.pth/","*"))
#wsi_list = glob(os.path.join("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/WGM_crops_gt1/","*"))
wsi_list = glob(os.path.join("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops1/3uctrn1o10.pth/","*"))
wsi_list.sort()
print(wsi_list)
for wsi in wsi_list[1:]:
    img_crop_list = glob(os.path.join(wsi,"grey","*.png"))
    print(img_crop_list)
    obj_coords_all = run_lb_seg(img_crop_list, wsi)
    break

['/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops1/3uctrn1o10.pth/04_028_Syn1_CG_200x.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops1/3uctrn1o10.pth/14_133_FCx_aSyn_x200.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops1/3uctrn1o10.pth/15_134_FCx_aSyn_x200.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops1/3uctrn1o10.pth/16_044_PCx_aSyn_x200.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops1/3uctrn1o10.pth/19_053_TCx_aSyn_x200.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops1/3uctrn1o10.pth/PD110_Syn1_TCx.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops1/3uctrn1o10.pth/PD125_Syn1_PCx.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD

100%|██████████| 1/1 [00:01<00:00,  1.16s/it]


[129.74673077584742, 133.50032799967056, 23.88666659331616]


100%|██████████| 1/1 [00:00<00:00,  9.58it/s]


[105.5689943351552, 155.53242844817382, 133.75173666848286]


...

100%|██████████| 1/1 [00:00<00:00, 10.76it/s]


[45.91259392399847, 128.39812840355307, 83.05468138815591]


100%|██████████| 1/1 [00:00<00:00, 10.77it/s]


[66.4520810847228, 237.63380547192386, 165.89217812348392]


100%|██████████| 1/1 [00:00<00:00, 10.78it/s]


In [21]:
obj_coords_all